In [312]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import os


### Load dataset

In [313]:
dir = "playground-series-s6e7"
train_folder = os.path.join(dir, "train.csv")
test_folder = os.path.join(dir, "test.csv")

In [314]:
train_data = pd.read_csv(train_folder, index_col="id")
test_data = pd.read_csv(test_folder, index_col="id")
train_data.shape, test_data.shape

((690088, 14), (295753, 13))

In [315]:
train_data.head()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,,
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [316]:
test_data.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


### EDA & data preprocessing

#### data Preview

In [317]:
train_data.head()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,,
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [318]:
train_data.tail()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,,
690083,at-risk,6.31,69.7,24.11,2157.0,NaN,30.8,3.00,non-veg,high,poor,active,yes,female
690084,at-risk,5.78,54.0,26.36,2858.0,6488.0,52.4,1.46,veg,medium,average,moderate,no,male
690085,fit,7.64,85.7,21.91,2195.0,9241.0,41.3,1.57,non-veg,NaN,average,active,no,male
690086,at-risk,6.74,73.0,18.73,2061.0,13366.0,56.6,2.60,balanced,NaN,average,active,yes,male
690087,at-risk,5.55,69.3,24.38,2257.0,5144.0,47.9,1.76,balanced,medium,average,moderate,yes,male


In [319]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  str    
 1   sleep_duration           614089 non-null  float64
 2   heart_rate               682255 non-null  float64
 3   bmi                      676190 non-null  float64
 4   calorie_expenditure      637235 non-null  float64
 5   step_count               676172 non-null  float64
 6   exercise_duration        683187 non-null  float64
 7   water_intake             646611 non-null  float64
 8   diet_type                683187 non-null  str    
 9   stress_level             607277 non-null  str    
 10  sleep_quality            631757 non-null  str    
 11  physical_activity_level  653467 non-null  str    
 12  smoking_alcohol          661506 non-null  str    
 13  gender                   668715 non-null  str    
dtypes: float64(7), 

In [320]:
train_data.describe(include=["str", "float64"])

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
count,690088,614089.000000,682255.000000,676190.000000,637235.000000,676172.000000,683187.000000,646611.000000,683187,607277,631757,653467,661506,668715
unique,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,3,3
top,at-risk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,veg,medium,average,moderate,yes,male
freq,592561,NaN,NaN,NaN,NaN,NaN,NaN,NaN,231432,261819,213948,221041,223730,237756
mean,NaN,6.992597,75.096504,22.984925,2226.084931,8615.953050,38.751456,2.188542,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,1.215407,8.175106,2.481787,347.532098,3929.399831,14.742189,0.518489,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,3.000000,50.000000,16.000000,1200.000000,1002.000000,0.000000,0.500000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,6.160000,69.400000,21.320000,2053.000000,5389.000000,29.200000,1.840000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,6.990000,75.100000,22.990000,2241.000000,8856.000000,39.400000,2.170000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,7.810000,80.700000,24.660000,2456.000000,12114.000000,49.400000,2.500000,NaN,NaN,NaN,NaN,NaN,NaN


#### Dealing with Non-Numerical data

In [321]:
category_columns = train_data.select_dtypes("str").columns
test_cat_columns = list(category_columns)
test_cat_columns.remove("health_condition")
category_columns

Index(['health_condition', 'diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender'],
      dtype='str')

In [322]:
train_data[category_columns] = train_data[category_columns].astype(np.object_)

In [323]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  object 
 1   sleep_duration           614089 non-null  float64
 2   heart_rate               682255 non-null  float64
 3   bmi                      676190 non-null  float64
 4   calorie_expenditure      637235 non-null  float64
 5   step_count               676172 non-null  float64
 6   exercise_duration        683187 non-null  float64
 7   water_intake             646611 non-null  float64
 8   diet_type                683187 non-null  object 
 9   stress_level             607277 non-null  object 
 10  sleep_quality            631757 non-null  object 
 11  physical_activity_level  653467 non-null  object 
 12  smoking_alcohol          661506 non-null  object 
 13  gender                   668715 non-null  object 
dtypes: float64(7), 

In [324]:
train_data[category_columns].isnull().sum()

health_condition               0
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [325]:
mode_cat = train_data[category_columns].mode().loc[0]
mode_cat

health_condition            at-risk
diet_type                       veg
stress_level                 medium
sleep_quality               average
physical_activity_level    moderate
smoking_alcohol                 yes
gender                         male
Name: 0, dtype: object

In [326]:
train_data[category_columns] = train_data[category_columns].fillna(mode_cat)
test_data[test_cat_columns] = test_data[test_cat_columns].fillna(mode_cat)
train_data[category_columns].isnull().sum()

health_condition           0
diet_type                  0
stress_level               0
sleep_quality              0
physical_activity_level    0
smoking_alcohol            0
gender                     0
dtype: int64

#### Dealing with Numerical Data

In [327]:
numerical_columns = train_data.select_dtypes("float64").columns
numerical_columns

Index(['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake'],
      dtype='str')

In [328]:
train_data[numerical_columns]

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
id,,,,,,,
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25
...,...,...,...,...,...,...,...
690083,6.31,69.7,24.11,2157.0,NaN,30.8,3.00
690084,5.78,54.0,26.36,2858.0,6488.0,52.4,1.46
690085,7.64,85.7,21.91,2195.0,9241.0,41.3,1.57


In [329]:
train_data[numerical_columns].isnull().sum()

sleep_duration         75999
heart_rate              7833
bmi                    13898
calorie_expenditure    52853
step_count             13916
exercise_duration       6901
water_intake           43477
dtype: int64

In [330]:
train_data[numerical_columns].mode().loc[0]

sleep_duration             7.16
heart_rate                74.00
bmi                       23.44
calorie_expenditure     2201.00
step_count             12182.00
exercise_duration          0.00
water_intake               2.37
Name: 0, dtype: float64

In [331]:
train_data[numerical_columns].describe()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
count,614089.000000,682255.000000,676190.000000,637235.000000,676172.000000,683187.000000,646611.000000
mean,6.992597,75.096504,22.984925,2226.084931,8615.953050,38.751456,2.188542
std,1.215407,8.175106,2.481787,347.532098,3929.399831,14.742189,0.518489
min,3.000000,50.000000,16.000000,1200.000000,1002.000000,0.000000,0.500000
25%,6.160000,69.400000,21.320000,2053.000000,5389.000000,29.200000,1.840000
50%,6.990000,75.100000,22.990000,2241.000000,8856.000000,39.400000,2.170000
75%,7.810000,80.700000,24.660000,2456.000000,12114.000000,49.400000,2.500000
max,10.000000,107.700000,34.820000,3580.000000,14999.000000,99.800000,4.720000


In [332]:
mean_fill_num = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure", "step_count", "water_intake"]
mode_fill_num = ["exercise_duration"]

In [333]:
mean_num = train_data[mean_fill_num].mean()
mode_num = train_data[mode_fill_num].mode().loc[0]
mode_num

exercise_duration    0.0
Name: 0, dtype: float64

In [334]:
train_data[mean_fill_num] = train_data[mean_fill_num].fillna(mean_num)
train_data[mode_fill_num] = train_data[mode_fill_num].fillna(mode_num)

test_data[mean_fill_num] = test_data[mean_fill_num].fillna(mean_num)
test_data[mode_fill_num] = test_data[mode_fill_num].fillna(mode_num)

In [335]:
train_data[numerical_columns].isnull().sum()

sleep_duration         0
heart_rate             0
bmi                    0
calorie_expenditure    0
step_count             0
exercise_duration      0
water_intake           0
dtype: int64

In [336]:
train_data.isnull().sum()

health_condition           0
sleep_duration             0
heart_rate                 0
bmi                        0
calorie_expenditure        0
step_count                 0
exercise_duration          0
water_intake               0
diet_type                  0
stress_level               0
sleep_quality              0
physical_activity_level    0
smoking_alcohol            0
gender                     0
dtype: int64

#### Encode Categorical data

In [337]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  object 
 1   sleep_duration           690088 non-null  float64
 2   heart_rate               690088 non-null  float64
 3   bmi                      690088 non-null  float64
 4   calorie_expenditure      690088 non-null  float64
 5   step_count               690088 non-null  float64
 6   exercise_duration        690088 non-null  float64
 7   water_intake             690088 non-null  float64
 8   diet_type                690088 non-null  object 
 9   stress_level             690088 non-null  object 
 10  sleep_quality            690088 non-null  object 
 11  physical_activity_level  690088 non-null  object 
 12  smoking_alcohol          690088 non-null  object 
 13  gender                   690088 non-null  object 
dtypes: float64(7), 

In [338]:
train_data.head(3)

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,,
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male


In [339]:
train_data.describe(include=[np.object_])

,health_condition,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
count,690088,690088,690088,690088,690088,690088,690088
unique,3,3,3,3,3,3,3
top,at-risk,veg,medium,average,moderate,yes,male
freq,592561,238333,344630,272279,257662,252312,259129


In [340]:
encode_map = {}
for col in category_columns:
    val_c = list(train_data[col].value_counts().to_dict().keys())
    encode_map[col] = {c: i for i, c in enumerate(val_c)}
    train_data[col] = train_data[col].replace(encode_map[col])
    if col != "health_condition":
        test_data[col] = test_data[col].replace(encode_map[col])

print(encode_map)



{'health_condition': {'at-risk': 0, 'unhealthy': 1, 'fit': 2}, 'diet_type': {'veg': 0, 'balanced': 1, 'non-veg': 2}, 'stress_level': {'medium': 0, 'high': 1, 'low': 2}, 'sleep_quality': {'average': 0, 'poor': 1, 'good': 2}, 'physical_activity_level': {'moderate': 0, 'sedentary': 1, 'active': 2}, 'smoking_alcohol': {'yes': 0, 'no': 1, 'occasional': 2}, 'gender': {'male': 0, 'female': 1, 'other': 2}}


In [341]:
train_data[category_columns]

,health_condition,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,
0,1,0,1,0,1,0,1
1,0,2,2,0,0,0,2
2,1,0,1,1,2,0,0
3,1,0,1,0,2,2,1
4,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...
690083,0,2,1,1,2,0,1
690084,0,0,0,0,0,1,0
690085,2,2,0,0,2,1,0


In [354]:
train_data = train_data.astype("float64")
test_data = test_data.astype("float64")
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  float64
 1   sleep_duration           690088 non-null  float64
 2   heart_rate               690088 non-null  float64
 3   bmi                      690088 non-null  float64
 4   calorie_expenditure      690088 non-null  float64
 5   step_count               690088 non-null  float64
 6   exercise_duration        690088 non-null  float64
 7   water_intake             690088 non-null  float64
 8   diet_type                690088 non-null  float64
 9   stress_level             690088 non-null  float64
 10  sleep_quality            690088 non-null  float64
 11  physical_activity_level  690088 non-null  float64
 12  smoking_alcohol          690088 non-null  float64
 13  gender                   690088 non-null  float64
dtypes: float64(14)


#### Normalize Numerical data

In [343]:
train_data[numerical_columns]

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
id,,,,,,,
0,5.22,70.6,25.66,2174.0,1326.00000,19.8,1.86
1,5.53,71.3,25.84,1966.0,9891.00000,49.9,1.26
2,5.29,75.4,24.54,2688.0,14216.00000,38.1,1.60
3,4.70,77.2,23.13,2630.0,7174.00000,59.9,2.02
4,7.23,73.4,28.44,2560.0,6584.00000,46.0,2.25
...,...,...,...,...,...,...,...
690083,6.31,69.7,24.11,2157.0,8615.95305,30.8,3.00
690084,5.78,54.0,26.36,2858.0,6488.00000,52.4,1.46
690085,7.64,85.7,21.91,2195.0,9241.00000,41.3,1.57


In [344]:
train_data.max()

health_condition               2.00
sleep_duration                10.00
heart_rate                   107.70
bmi                           34.82
calorie_expenditure         3580.00
step_count                 14999.00
exercise_duration             99.80
water_intake                   4.72
diet_type                      2.00
stress_level                   2.00
sleep_quality                  2.00
physical_activity_level        2.00
smoking_alcohol                2.00
gender                         2.00
dtype: float64

In [345]:
train_data.min()

health_condition              0.0
sleep_duration                3.0
heart_rate                   50.0
bmi                          16.0
calorie_expenditure        1200.0
step_count                 1002.0
exercise_duration             0.0
water_intake                  0.5
diet_type                     0.0
stress_level                  0.0
sleep_quality                 0.0
physical_activity_level       0.0
smoking_alcohol               0.0
gender                        0.0
dtype: float64

In [346]:
test_data[numerical_columns] = (test_data[numerical_columns] - train_data[numerical_columns].min()) / (train_data[numerical_columns].max() - train_data[numerical_columns].min())

train_data[numerical_columns] = (train_data[numerical_columns] - train_data[numerical_columns].min()) / (train_data[numerical_columns].max() - train_data[numerical_columns].min())

train_data[numerical_columns]

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
id,,,,,,,
0,0.317143,0.357019,0.513284,0.409244,0.023148,0.198397,0.322275
1,0.361429,0.369151,0.522848,0.321849,0.635065,0.500000,0.180095
2,0.327143,0.440208,0.453773,0.625210,0.944059,0.381764,0.260664
3,0.242857,0.471404,0.378852,0.600840,0.440952,0.600200,0.360190
4,0.604286,0.405546,0.660999,0.571429,0.398800,0.460922,0.414692
...,...,...,...,...,...,...,...
690083,0.472857,0.341421,0.430925,0.402101,0.543970,0.308617,0.592417
690084,0.397143,0.069324,0.550478,0.696639,0.391941,0.525050,0.227488
690085,0.662857,0.618718,0.314028,0.418067,0.588626,0.413828,0.253555


In [347]:
train_data

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,,
0,1.0,0.317143,0.357019,0.513284,0.409244,0.023148,0.198397,0.322275,0.0,1.0,0.0,1.0,0.0,1.0
1,0.0,0.361429,0.369151,0.522848,0.321849,0.635065,0.500000,0.180095,2.0,2.0,0.0,0.0,0.0,2.0
2,1.0,0.327143,0.440208,0.453773,0.625210,0.944059,0.381764,0.260664,0.0,1.0,1.0,2.0,0.0,0.0
3,1.0,0.242857,0.471404,0.378852,0.600840,0.440952,0.600200,0.360190,0.0,1.0,0.0,2.0,2.0,1.0
4,0.0,0.604286,0.405546,0.660999,0.571429,0.398800,0.460922,0.414692,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690083,0.0,0.472857,0.341421,0.430925,0.402101,0.543970,0.308617,0.592417,2.0,1.0,1.0,2.0,0.0,1.0
690084,0.0,0.397143,0.069324,0.550478,0.696639,0.391941,0.525050,0.227488,0.0,0.0,0.0,0.0,1.0,0.0
690085,2.0,0.662857,0.618718,0.314028,0.418067,0.588626,0.413828,0.253555,2.0,0.0,0.0,2.0,1.0,0.0


In [348]:
test_data[numerical_columns]

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
id,,,,,,,
690088,0.335714,0.258232,0.397450,0.649160,0.940559,0.596192,0.322275
690089,0.570371,0.573657,0.341126,0.240756,0.414303,0.245491,0.450237
690090,0.525714,0.168111,0.432519,0.773109,0.875045,0.485972,0.535545
690091,0.590000,0.493934,0.545165,0.543697,0.380724,0.570140,0.436019
690092,0.355714,0.480069,0.387354,0.263866,0.921055,0.394790,0.462085
...,...,...,...,...,...,...,...
985836,0.590000,0.476603,0.329437,0.428992,0.927270,0.657315,0.395735
985837,0.464286,0.337955,0.261955,0.437815,0.097735,0.236473,0.350711
985838,0.570371,0.467938,0.404888,0.382773,0.474745,0.251503,0.398104


In [349]:
y = train_data["health_condition"]
x = train_data[train_data.columns[1:]]
x

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
0,0.317143,0.357019,0.513284,0.409244,0.023148,0.198397,0.322275,0.0,1.0,0.0,1.0,0.0,1.0
1,0.361429,0.369151,0.522848,0.321849,0.635065,0.500000,0.180095,2.0,2.0,0.0,0.0,0.0,2.0
2,0.327143,0.440208,0.453773,0.625210,0.944059,0.381764,0.260664,0.0,1.0,1.0,2.0,0.0,0.0
3,0.242857,0.471404,0.378852,0.600840,0.440952,0.600200,0.360190,0.0,1.0,0.0,2.0,2.0,1.0
4,0.604286,0.405546,0.660999,0.571429,0.398800,0.460922,0.414692,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
690083,0.472857,0.341421,0.430925,0.402101,0.543970,0.308617,0.592417,2.0,1.0,1.0,2.0,0.0,1.0
690084,0.397143,0.069324,0.550478,0.696639,0.391941,0.525050,0.227488,0.0,0.0,0.0,0.0,1.0,0.0
690085,0.662857,0.618718,0.314028,0.418067,0.588626,0.413828,0.253555,2.0,0.0,0.0,2.0,1.0,0.0


### Model

In [375]:
modelA = keras.Sequential([
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(3, activation="softmax")
    ]
)
history = modelA.compile(
    optimizer="adam",
    loss=keras.metrics.sparse_categorical_crossentropy,
    metrics=["accuracy"]
)

In [376]:
modelA.fit(x, y, batch_size=512, epochs=10, validation_split=0.2)

Epoch 1/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9408 - loss: 0.2038 - val_accuracy: 0.9605 - val_loss: 0.1529
Epoch 2/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9592 - loss: 0.1509 - val_accuracy: 0.9617 - val_loss: 0.1435
Epoch 3/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9602 - loss: 0.1437 - val_accuracy: 0.9627 - val_loss: 0.1382
Epoch 4/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9608 - loss: 0.1391 - val_accuracy: 0.9630 - val_loss: 0.1347
Epoch 5/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9612 - loss: 0.1363 - val_accuracy: 0.9631 - val_loss: 0.1325
Epoch 6/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9614 - loss: 0.1345 - val_accuracy: 0.9633 - val_loss: 0.1306
Epoch 7/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9616 - loss: 0.1332 - val_accuracy: 0.9635 - val_loss: 0.1286
Epoch 8/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9616 - loss: 0.1320 - 

In [377]:
y_pred = modelA.predict(test_data)
y_pred

9243/9243 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step


array([[2.3047044e-03, 9.9153847e-01, 6.1567486e-03],
       [7.1280682e-01, 2.8486922e-01, 2.3239544e-03],
       [9.7756702e-01, 9.5603969e-03, 1.2872565e-02],
       ...,
       [9.0736437e-01, 8.9763001e-02, 2.8726554e-03],
       [9.8731494e-01, 1.0288985e-02, 2.3960883e-03],
       [4.8116161e-04, 9.9679095e-01, 2.7279193e-03]],
      shape=(295753, 3), dtype=float32)

In [408]:
prediction = pd.DataFrame(np.argmax(y_pred, axis=1), index=test_data.index)
prediction.columns = pd.Index(["health_condition"])
prediction = prediction.replace({i:j for j, i in encode_map["health_condition"].items()})
prediction

,health_condition
id,
690088,unhealthy
690089,at-risk
690090,at-risk
690091,at-risk
690092,unhealthy
...,...
985836,fit
985837,at-risk
985838,at-risk


In [409]:
prediction.to_csv("submission1.csv")